In [2]:
!pip install -q peft transformers datasets accelerate trl bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from google.colab import files
uploaded = files.upload()  # Upload your dataset.jsonl


Saving dataset.jsonl to dataset.jsonl


In [4]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
print(f"✅ Loaded {len(dataset)} examples")


Generating train split: 0 examples [00:00, ? examples/s]

✅ Loaded 458 examples


In [5]:
from transformers import AutoTokenizer

base_model = "gpt2-large"
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token  # GPT2 doesn't have a pad token


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
def tokenize_function(example):
    full_prompt = f"Instruction: {example['instruction']}\n{example['input']}\nPost: {example['output']}"
    tokenized = tokenizer(full_prompt, truncation=True, padding="max_length", max_length=150)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=False)
tokenized_dataset.set_format("torch")


Map:   0%|          | 0/458 [00:00<?, ? examples/s]

In [7]:
from transformers import AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

model = AutoModelForCausalLM.from_pretrained(base_model)
model.resize_token_embeddings(len(tokenizer))  # in case tokenizer resized

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 1,474,560 || all params: 775,504,640 || trainable%: 0.1901


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:1803: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-lora-checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    logging_steps=10,
    num_train_epochs=3,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,
    report_to="none",
)


In [9]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    peft_config=peft_config,
)

trainer.train()


Truncating train dataset:   0%|          | 0/458 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,5.473600
20,1.788600
30,1.448200
40,1.242300
50,1.057300
60,0.875200
70,0.898900
80,0.847600
90,0.813900
100,0.796400


TrainOutput(global_step=174, training_loss=1.2001764048105, metrics={'train_runtime': 86.906, 'train_samples_per_second': 15.81, 'train_steps_per_second': 2.002, 'total_flos': 877819009536000.0, 'train_loss': 1.2001764048105})

In [10]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    peft_config=peft_config,
)

trainer.train()


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,0.744300
20,0.723100
30,0.724700
40,0.720600
50,0.715500
60,0.640900
70,0.681500
80,0.652800
90,0.649900
100,0.645300


TrainOutput(global_step=174, training_loss=0.6639458461739551, metrics={'train_runtime': 87.5456, 'train_samples_per_second': 15.695, 'train_steps_per_second': 1.988, 'total_flos': 877819009536000.0, 'train_loss': 0.6639458461739551})

In [11]:
model.save_pretrained("./gpt2-lora-adapter")
tokenizer.save_pretrained("./gpt2-lora-adapter")


('./gpt2-lora-adapter/tokenizer_config.json',
 './gpt2-lora-adapter/special_tokens_map.json',
 './gpt2-lora-adapter/vocab.json',
 './gpt2-lora-adapter/merges.txt',
 './gpt2-lora-adapter/added_tokens.json',
 './gpt2-lora-adapter/tokenizer.json')

In [17]:
from transformers import pipeline
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
import nltk

# Ensure punkt is downloaded
nltk.download("punkt", download_dir="/usr/share/nltk_data")
import nltk.data
nltk.data.path.append("/usr/share/nltk_data")

# Set up generator
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

# Prompt parts
instruction = "Write a quirky social media post."
brand = "Starbucks"
news = "AI-designed traffic system reduces city congestion by 30%."
input_block = f"Brand: {brand}\nNews: {news}"
prompt = f"Instruction: {instruction}\n{input_block}\nPost:"

# Generate
output = generator(prompt, max_new_tokens=100, do_sample=True, temperature=0.9)[0]['generated_text']
print("📝 Prompt:\n", prompt)
print("\n🎯 Model Output:\n", output)

# Reference
reference = ["Traffic’s flowing smoother than cold brew on a Monday. Thanks, AI! 🚦☕️"]
candidate = output.split("Post:")[-1].strip()

# BLEU Score
score = sentence_bleu(
    [word_tokenize(reference[0])],
    word_tokenize(candidate),
    smoothing_function=SmoothingFunction().method4
)
print(f"\n🔍 BLEU Score: {score:.2f}")


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
Device set to use cuda:0


📝 Prompt:
 Instruction: Write a quirky social media post.
Brand: Starbucks
News: AI-designed traffic system reduces city congestion by 30%.
Post:

🎯 Model Output:
 Instruction: Write a quirky social media post.
Brand: Starbucks
News: AI-designed traffic system reduces city congestion by 30%.
Post: The world’s a better place with better coffee. 🔥☕🍕 #AIBusCoffee


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
    - '/usr/share/nltk_data'
**********************************************************************
